# HEDIS Analytics — Database Verification
Phase 4: verify all CMS synthetic Medicare tables loaded correctly into SQL Server.

Run these cells after any ETL reload to confirm row counts, date ranges, and data integrity.
All 9 tables should be present: beneficiary, inpatient, outpatient, carrier, dme, hha, hospice, snf, pde.

In [1]:
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
import getpass
from urllib.parse import quote_plus

password = getpass.getpass('SA password: ')

conn_str = (
    'DRIVER={ODBC Driver 18 for SQL Server};'
    'SERVER=localhost;'
    'DATABASE=hedis;'
    'UID=SA;'
    f'PWD={password};'
    'TrustServerCertificate=yes;'
)

engine = create_engine(f'mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}')
conn = engine.connect()
print('Connected.')

SA password:  ········


Connected.


## Row counts — verify all tables loaded
Expected counts based on source CSV files (lines minus 1 header row):

| Table | Expected rows |
|---|---|
| beneficiary | 8,246 (2021 file) |
| inpatient | 58,066 |
| outpatient | 575,092 |
| carrier | 1,121,004 |
| dme | 103,828 |
| hha | 6,215 |
| hospice | 12,107 |
| snf | 12,548 |
| pde | 515,520 |

In [2]:
tables = ['beneficiary', 'inpatient', 'outpatient', 'carrier', 'dme', 'hha', 'hospice', 'snf', 'pde']

counts = []
for t in tables:
    df = pd.read_sql(f'SELECT COUNT(*) AS row_count FROM {t}', conn)
    counts.append({'table': t, 'row_count': df['row_count'][0]})

pd.DataFrame(counts)

,table,row_count
0,beneficiary,8246
1,inpatient,58066
2,outpatient,575092
3,carrier,1121004
4,dme,103828
5,hha,6215
6,hospice,12107
7,snf,12548
8,pde,515520


## Beneficiary — sample rows
Spot check: BENE_ENROLLMT_REF_YR should be 2021. BENE_DEATH_DT should be NULL for most rows.

In [3]:
pd.read_sql("""
    SELECT TOP 5
        BENE_ID,
        BENE_BIRTH_DT,
        SEX_IDENT_CD,
        BENE_RACE_CD,
        BENE_DEATH_DT,
        BENE_ENROLLMT_REF_YR,
        STATE_CODE,
        ZIP_CD
    FROM beneficiary
""", conn)

,BENE_ID,BENE_BIRTH_DT,SEX_IDENT_CD,BENE_RACE_CD,BENE_DEATH_DT,BENE_ENROLLMT_REF_YR,STATE_CODE,ZIP_CD
0,-10000010254618,16-Aug-1999,1,1,None,2021,01,36109
1,-10000010254647,14-Aug-1990,1,2,None,2021,01,35756
2,-10000010254653,18-Mar-1982,2,1,None,2021,01,36801
3,-10000010254656,24-Jul-1999,2,1,None,2021,01,35216
4,-10000010254666,23-Feb-2009,2,4,None,2021,01,35805


## Beneficiary — age and sex distribution
Age calculated from BENE_BIRTH_DT as of Dec 31, 2021 (measurement year end).
Expect a roughly even male/female split and a wide age range skewed toward Medicare-eligible ages (65+).

In [4]:
pd.read_sql("""
    SELECT
        SEX_IDENT_CD,
        MIN(FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25)) AS min_age,
        MAX(FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25)) AS max_age,
        AVG(FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25)) AS avg_age,
        COUNT(*) AS bene_count
    FROM beneficiary
    WHERE BENE_DEATH_DT IS NULL
    GROUP BY SEX_IDENT_CD
    ORDER BY SEX_IDENT_CD
""", conn)

,SEX_IDENT_CD,min_age,max_age,avg_age,bene_count
0,1,0.0,109.0,60.201923,4160
1,2,0.0,109.0,60.545031,4086


## Claims — date range check
All claim tables should span April 2015 through October 2022.
Measurement year 2021 (Jan 1 – Dec 31) falls fully within this window.

In [5]:
pd.read_sql("""
    SELECT 'inpatient' AS claim_type, MIN(CLM_FROM_DT) AS earliest, MAX(CLM_THRU_DT) AS latest FROM inpatient
    UNION ALL
    SELECT 'outpatient', MIN(CLM_FROM_DT), MAX(CLM_THRU_DT) FROM outpatient
    UNION ALL
    SELECT 'carrier', MIN(CLM_FROM_DT), MAX(CLM_THRU_DT) FROM carrier
    UNION ALL
    SELECT 'pde', MIN(SRVC_DT), MAX(SRVC_DT) FROM pde
""", conn)

,claim_type,earliest,latest
0,inpatient,01-Apr-2015,31-Oct-2022
1,outpatient,01-Apr-2015,31-Oct-2022
2,carrier,01-Apr-2015,31-Oct-2022
3,pde,01-Apr-2015,31-Oct-2022


## Inpatient — sample diagnosis codes
Spot check: ICD-10 codes should be present and look valid (letter + digits format).

In [6]:
pd.read_sql("""
    SELECT TOP 10
        BENE_ID,
        CLM_FROM_DT,
        CLM_THRU_DT,
        PRNCPAL_DGNS_CD,
        ICD_DGNS_CD1,
        ICD_DGNS_CD2,
        ICD_DGNS_CD3
    FROM inpatient
    WHERE CLM_LINE_NUM = 1
""", conn)

,BENE_ID,CLM_FROM_DT,CLM_THRU_DT,PRNCPAL_DGNS_CD,ICD_DGNS_CD1,ICD_DGNS_CD2,ICD_DGNS_CD3
0,-10000010254618,25-Mar-2015,25-Mar-2015,S134XX,S134XX,R4689,E781
1,-10000010254653,24-Sep-2015,24-Sep-2015,Z3480,T7432X,E669,C50919
2,-10000010254653,09-May-2017,10-May-2017,T7432X,T7432X,E669,C50929
3,-10000010254656,14-Jan-2017,14-Jan-2017,S8290X,S8290X,G40909,R569
4,-10000010254656,17-Mar-2018,17-Mar-2018,Z3480,Z5989,Z5941,G40909
5,-10000010254656,12-Mar-2022,12-Mar-2022,Z3400,T7432X,Z733,Z5989
6,-10000010254666,21-Mar-2015,21-Mar-2015,R569,G40909,P90,Z8669
7,-10000010254682,30-Apr-2021,30-Apr-2021,S8290X,S8290X,Z733,I252
8,-10000010254691,12-Mar-2020,12-Mar-2020,Z608,Z608,Z7682,N184
9,-10000010254691,21-May-2020,21-May-2020,Z608,Z608,Z7682,N184


## Carrier — sample HCPCS codes
Spot check: HCPCS codes should be present. Carrier claims are physician and outpatient services billed under Part B.

In [7]:
pd.read_sql("""
    SELECT TOP 10
        BENE_ID,
        CLM_FROM_DT,
        HCPCS_CD,
        LINE_ICD_DGNS_CD,
        LINE_NCH_PMT_AMT
    FROM carrier
    WHERE HCPCS_CD IS NOT NULL
""", conn)

,BENE_ID,CLM_FROM_DT,HCPCS_CD,LINE_ICD_DGNS_CD,LINE_NCH_PMT_AMT
0,-10000010254618,28-Sep-2015,96127,None,367.28
1,-10000010254618,28-Sep-2015,G0444,R4689,0.00
2,-10000010254618,28-Sep-2015,G9573,None,0.00
3,-10000010254618,03-Oct-2016,96127,R4689,367.28
4,-10000010254618,03-Oct-2016,G0444,None,0.00
5,-10000010254618,03-Oct-2016,G9573,R4689,0.00
6,-10000010254618,03-Oct-2016,99408,R4689,0.00
7,-10000010254618,09-Oct-2017,99495,R4689,302.24
8,-10000010254618,09-Oct-2017,96156,None,367.28
9,-10000010254618,09-Oct-2017,96127,None,0.00


## PDE — sample drug records
Spot check: PROD_SRVC_ID (NDC code), DAYS_SUPLY_NUM, and TOT_RX_CST_AMT should all be populated.
DRUG_CVRG_STUS_CD = 'C' indicates covered under Part D.

In [8]:
pd.read_sql("""
    SELECT TOP 10
        BENE_ID,
        SRVC_DT,
        PROD_SRVC_ID,
        DAYS_SUPLY_NUM,
        TOT_RX_CST_AMT,
        DRUG_CVRG_STUS_CD
    FROM pde
""", conn)

,BENE_ID,SRVC_DT,PROD_SRVC_ID,DAYS_SUPLY_NUM,TOT_RX_CST_AMT,DRUG_CVRG_STUS_CD
0,-10000010254618,25-Mar-2015,68115025030,63,275.19,C
1,-10000010254618,27-May-2016,53978010903,7,2420.67,C
2,-10000010254618,03-Oct-2016,55154010000,90,1204.91,C
3,-10000010254618,20-Sep-2017,13107021199,10,136.48,C
4,-10000010254618,30-Sep-2017,13107021199,10,136.48,C
5,-10000010254618,09-Oct-2017,11014070601,90,1388.21,C
6,-10000010254618,09-Oct-2017,10702019250,10,104.13,C
7,-10000010254618,19-Oct-2017,10702019250,10,104.13,C
8,-10000010254618,29-Oct-2017,10702019250,10,104.13,C
9,-10000010254618,08-Nov-2017,10702019250,10,104.13,C
